In [7]:
import pandas as pd

In [8]:
df = pd.read_csv('./df_all_stage_ready.csv')
df.head()

,Question,Source Docs,Question Type,Source Chunk Type,Answer,spelling_errors_text,keyboard_typos_text,word_deletion_text,token_permutation_text,listed_sources
0,How has Apple's total net sales changed over t...,*AAPL*,Multi-Doc RAG,Table,"Based on the provided documents, Apple's total...",How has Apple's totul net sales changid over t...,How has Apple's total net sales changed over r...,How has Apple's net sales changed over time?,has How Apple' total s sales net over changed ...,"['2022 Q3 AAPL.pdf', '2023 Q1 AAPL.pdf', '2023..."
1,What are the major factors contributing to the...,*AAPL*,Multi-Doc RAG,Text,In the most recent 10-Q for the quarter ended ...,Whit are the major factors contributin to the ...,What are the kajor factors contributing to the...,What are the major factors contributing to the...,are What the major contributing factors the to...,['2023 Q3 AAPL.pdf']
2,Has there been any significant change in Apple...,*AAPL*,Multi-Doc RAG,Table,"Yes, there has been a change in Apple's operat...",Has thure been any significant chinge in Apple...,Has there heen any significant change in Apple...,Has there been any significant change in 's op...,there Has been any change significant Apple in...,"['2022 Q3 AAPL.pdf', '2023 Q1 AAPL.pdf', '2023..."
3,How has Apple's revenue from iPhone sales fluc...,*AAPL*,Multi-Doc RAG,Table,The revenue from iPhone sales for Apple has fl...,How has Apple's revenue frum ifone sales fluct...,How has Apple's fevenue from iPhone sales fluc...,How has Apple's revenue from sales fluctuated...,has How Apple' revenue s iPhone from fluctuate...,"['2022 Q3 AAPL.pdf', '2023 Q1 AAPL.pdf', '2023..."
4,Can any trends be identified in Apple's Servic...,*AAPL*,Multi-Doc RAG,Table,"Based on the provided documents, there is a tr...",Can any trends be identified in Apple's Servac...,Can any trends be kdentified in Apple's Servic...,Can any trends be identified in Apple's Servic...,any Can trends be in identified' Apple Service...,"['2022 Q3 AAPL.pdf', '2023 Q1 AAPL.pdf', '2023..."


## Генерация ошибочных датасетов

### Орфографические ошибки (stage 1)

In [31]:
import random
import re

MONTHS = {
    "january", "february", "march", "april", "may", "june",
    "july", "august", "september", "october", "november", "december"
}

PHONETIC_RULES = [
    ("all", "oll"),
    ("al", "ol"),
    ("ed", "id"),
    ("ing", "in"),
    ("ou", "u"),
    ("ie", "ei"),
    ("ph", "f"),
    ("ck", "k"),
    ("tion", "shun"),
]

VOWELS = "aeiou"


WORD_RE = re.compile(r"\b[A-Za-z]+\b")


def preserve_case(original: str, new: str) -> str:

    if original.istitle():
        return new.capitalize()

    if original.isupper():
        return new.upper()

    return new


def phonetic_typo(word: str) -> str:

    lower = word.lower()

    # 1. phonetic substitutions
    possible = []

    for src, dst in PHONETIC_RULES:
        if src in lower:
            possible.append((src, dst))

    if possible and random.random() < 0.7:

        src, dst = random.choice(possible)

        idx = lower.find(src)

        new_word = (
            word[:idx]
            + dst
            + word[idx + len(src):]
        )

        return preserve_case(word, new_word)

    # 2. vowel confusion
    vowel_positions = [
        i for i, ch in enumerate(lower)
        if ch in VOWELS
    ]

    if vowel_positions:

        idx = random.choice(vowel_positions)

        replacement = random.choice(VOWELS)

        while replacement == lower[idx]:
            replacement = random.choice(VOWELS)

        chars = list(word)
        chars[idx] = replacement

        return preserve_case(word, "".join(chars))

    return word


def should_skip(word: str) -> bool:

    if len(word) < 4:
        return True

    if any(ch.isdigit() for ch in word):
        return True

    if "'" in word:
        return True

    if word.lower() in MONTHS:
        return True

    if word.lower().endswith(".pdf"):
        return True

    if word.isupper():
        return True

    return False


def corrupt_text(
    text: str,
    corruption_rate: float = 0.2,
    seed: int = None
) -> str:

    if seed is not None:
        random.seed(seed)

    lines = text.splitlines()

    result_lines = []

    for line in lines:

        # SOURCE строки не трогаем
        if "SOURCE:" in line or "SOURCE(S):" in line:
            result_lines.append(line)
            continue

        matches = list(WORD_RE.finditer(line))

        candidates = []

        for m in matches:

            word = m.group()

            if should_skip(word):
                continue

            candidates.append(m)

        num_to_corrupt = int(len(candidates) * corruption_rate)

        if candidates and num_to_corrupt == 0:
            num_to_corrupt = 1

        selected = set(
            random.sample(
                range(len(candidates)),
                min(num_to_corrupt, len(candidates))
            )
        )

        new_line = line

        replacements = []

        for i, match in enumerate(candidates):

            if i not in selected:
                continue

            original = match.group()

            corrupted = phonetic_typo(original)

            replacements.append((
                match.start(),
                match.end(),
                corrupted
            ))

        # заменяем справа налево
        for start, end, corrupted in reversed(replacements):

            new_line = (
                new_line[:start]
                + corrupted
                + new_line[end:]
            )

        result_lines.append(new_line)

    return "\n".join(result_lines)

In [37]:
text = """
Based on the provided documents, Apple's total net sales have changed over time as follows:

- For the quarterly period ended June 25, 2022, the total net sales were $82,959 million.

From these figures, it can be observed that there was an increase in total net sales.

SOURCE(S): 2022 Q3 AAPL.pdf
"""

result = corrupt_text(
    text,
    corruption_rate=0.35,
    seed=97
)

print(result)


Based on the provided documents, Upple's total net soles hive changed over time as follows:

- For the qoarterly period endid June 25, 2022, the total net sales were $82,959 million.

From these fogures, it can be observid that thure was an increase in total net sales.

SOURCE(S): 2022 Q3 AAPL.pdf


In [43]:
df["spelling_errors_text"] = [
    corrupt_text(text, corruption_rate=0.4, seed=97)
    for text in df["Question"]]
df

,Question,Source Docs,Question Type,Source Chunk Type,Answer,spelling_errors_text
0,How has Apple's total net sales changed over t...,*AAPL*,Multi-Doc RAG,Table,"Based on the provided documents, Apple's total...",How has Apple's totul net sales changid over t...
1,What are the major factors contributing to the...,*AAPL*,Multi-Doc RAG,Text,In the most recent 10-Q for the quarter ended ...,Whit are the major factors contributin to the ...
2,Has there been any significant change in Apple...,*AAPL*,Multi-Doc RAG,Table,"Yes, there has been a change in Apple's operat...",Has thure been any significant chinge in Apple...
3,How has Apple's revenue from iPhone sales fluc...,*AAPL*,Multi-Doc RAG,Table,The revenue from iPhone sales for Apple has fl...,How has Apple's revenue frum ifone sales fluct...
4,Can any trends be identified in Apple's Servic...,*AAPL*,Multi-Doc RAG,Table,"Based on the provided documents, there is a tr...",Can any trends be identified in Apple's Servac...
...,...,...,...,...,...,...
190,"For Amazon's Q1 2023 10-Q, align the details o...",*2023 Q1 AMZN*,Single-Doc Multi-Chunk RAG,Table,"In Amazon's Q1 2023 10-Q, the details of debt ...","For Amazan's Q1 2023 10-Q, align the details o..."
191,Analyze how Amazon's effective tax rate report...,*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The effective tax rate for Amazon as reported ...,Anolyze how Amazon's effective tax rite report...
192,"From Amazon's Q3 2023 10-Q, how does the opera...",*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The operational expenses section in Amazon's Q...,"Frum Amazon's Q3 2023 10-Q, how does the ipera..."
193,"In the latest 10-Q, how does the revenue from ...",*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The latest 10-Q does not provide specific info...,"In the litest 10-Q, how does the revenue fram ..."


### Ошибки соседний клавиш  (stage 2)

In [45]:
import random
import re

MONTHS = {
    "january", "february", "march", "april", "may", "june",
    "july", "august", "september", "october", "november", "december"
}

# Соседние клавиши QWERTY
KEYBOARD_NEIGHBORS = {
    "a": "qwsz",
    "b": "vghn",
    "c": "xdfv",
    "d": "serfcx",
    "e": "wsdfr",
    "f": "drtgvc",
    "g": "ftyhbv",
    "h": "gyujnb",
    "i": "ujko",
    "j": "huikmn",
    "k": "jiolm",
    "l": "kop",
    "m": "njk",
    "n": "bhjm",
    "o": "iklp",
    "p": "ol",
    "q": "wa",
    "r": "edft",
    "s": "awedxz",
    "t": "rfgy",
    "u": "yhji",
    "v": "cfgb",
    "w": "qase",
    "x": "zsdc",
    "y": "tghu",
    "z": "asx",
}


WORD_RE = re.compile(r"\b[A-Za-z]+\b")


def preserve_case(original: str, new: str) -> str:

    if original.istitle():
        return new.capitalize()

    if original.isupper():
        return new.upper()

    return new


def keyboard_typo(word: str) -> str:
    """
    Реалистичная ошибка соседней клавиши.
    """

    lower = word.lower()

    valid_positions = [
        i for i, ch in enumerate(lower)
        if ch in KEYBOARD_NEIGHBORS
    ]

    if not valid_positions:
        return word

    idx = random.choice(valid_positions)

    original_char = word[idx]
    lower_char = lower[idx]

    replacement = random.choice(
        KEYBOARD_NEIGHBORS[lower_char]
    )

    # сохраняем регистр
    if original_char.isupper():
        replacement = replacement.upper()

    typo = (
        word[:idx]
        + replacement
        + word[idx + 1:]
    )

    return typo


def should_skip(word: str) -> bool:

    if len(word) < 4:
        return True

    if any(ch.isdigit() for ch in word):
        return True

    # Apple's
    if "'" in word:
        return True

    if word.lower() in MONTHS:
        return True

    if word.lower().endswith(".pdf"):
        return True

    if word.isupper():
        return True

    return False


def corrupt_text(
    text: str,
    corruption_rate: float = 0.2,
    seed: int = None
) -> str:

    if seed is not None:
        random.seed(seed)

    lines = text.splitlines()

    result_lines = []

    for line in lines:

        # SOURCE строки не трогаем
        if "SOURCE:" in line or "SOURCE(S):" in line:
            result_lines.append(line)
            continue

        matches = list(WORD_RE.finditer(line))

        candidates = []

        for match in matches:

            word = match.group()

            if should_skip(word):
                continue

            candidates.append(match)

        num_to_corrupt = int(
            len(candidates) * corruption_rate
        )

        if candidates and num_to_corrupt == 0:
            num_to_corrupt = 1

        selected = set(
            random.sample(
                range(len(candidates)),
                min(num_to_corrupt, len(candidates))
            )
        )

        new_line = line

        replacements = []

        for i, match in enumerate(candidates):

            if i not in selected:
                continue

            original = match.group()

            corrupted = keyboard_typo(original)

            replacements.append((
                match.start(),
                match.end(),
                corrupted
            ))

        # справа налево
        for start, end, corrupted in reversed(replacements):

            new_line = (
                new_line[:start]
                + corrupted
                + new_line[end:]
            )

        result_lines.append(new_line)

    return "\n".join(result_lines)

In [49]:
text = """
Based on the provided documents, Apple's total net sales have changed over time as follows:

- For the quarterly period ended June 25, 2022, the total net sales were $82,959 million.

From these figures, it can be observed that there was an increase in total net sales.

SOURCE(S): 2022 Q3 AAPL.pdf
"""

result = corrupt_text(
        text,
        corruption_rate=0.2,
        seed=42
    )

print(result)


Based on the lrovided documents, Apple's total net sales have changed over time as fkllows:

- For the quarterly perios ended June 25, 2022, the total net sales were $82,959 million.

From these figures, it can be observed that there was an increase in total net xales.

SOURCE(S): 2022 Q3 AAPL.pdf


In [50]:
df["keyboard_typos_text"] = [
    corrupt_text(text, corruption_rate=0.2, seed=42)
    for text in df["Question"]]
df

,Question,Source Docs,Question Type,Source Chunk Type,Answer,spelling_errors_text,keyboard_typos_text
0,How has Apple's total net sales changed over t...,*AAPL*,Multi-Doc RAG,Table,"Based on the provided documents, Apple's total...",How has Apple's totul net sales changid over t...,How has Apple's total net sales changed over r...
1,What are the major factors contributing to the...,*AAPL*,Multi-Doc RAG,Text,In the most recent 10-Q for the quarter ended ...,Whit are the major factors contributin to the ...,What are the kajor factors contributing to the...
2,Has there been any significant change in Apple...,*AAPL*,Multi-Doc RAG,Table,"Yes, there has been a change in Apple's operat...",Has thure been any significant chinge in Apple...,Has there heen any significant change in Apple...
3,How has Apple's revenue from iPhone sales fluc...,*AAPL*,Multi-Doc RAG,Table,The revenue from iPhone sales for Apple has fl...,How has Apple's revenue frum ifone sales fluct...,How has Apple's fevenue from iPhone sales fluc...
4,Can any trends be identified in Apple's Servic...,*AAPL*,Multi-Doc RAG,Table,"Based on the provided documents, there is a tr...",Can any trends be identified in Apple's Servac...,Can any trends be kdentified in Apple's Servic...
...,...,...,...,...,...,...,...
190,"For Amazon's Q1 2023 10-Q, align the details o...",*2023 Q1 AMZN*,Single-Doc Multi-Chunk RAG,Table,"In Amazon's Q1 2023 10-Q, the details of debt ...","For Amazan's Q1 2023 10-Q, align the details o...","For Amazon's Q1 2023 10-Q, slign the details o..."
191,Analyze how Amazon's effective tax rate report...,*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The effective tax rate for Amazon as reported ...,Anolyze how Amazon's effective tax rite report...,Analyze how Smazon's effective tax rate report...
192,"From Amazon's Q3 2023 10-Q, how does the opera...",*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The operational expenses section in Amazon's Q...,"Frum Amazon's Q3 2023 10-Q, how does the ipera...","From Smazon's Q3 2023 10-Q, how does the opera..."
193,"In the latest 10-Q, how does the revenue from ...",*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The latest 10-Q does not provide specific info...,"In the litest 10-Q, how does the revenue fram ...","In the latest 10-Q, how xoes the revenue from ..."


### Пропуск слов  (stage 3)

In [1]:
def random_word_deletion(text: str, rate: float = 0.2) -> str:
    """
    Удаляет ~rate слов (word omission),
    сохраняя структуру текста.
    """

    lines = text.splitlines()
    result_lines = []

    for line in lines:

        # SOURCE строки не трогаем
        if "SOURCE:" in line or "SOURCE(S):" in line:
            result_lines.append(line)
            continue

        matches = list(WORD_RE.finditer(line))

        candidates = []

        for m in matches:
            word = m.group()

            if len(word) < 4:
                continue

            if any(ch.isdigit() for ch in word):
                continue

            if "'" in word:
                continue

            if word.lower() in MONTHS:
                continue

            if word.lower().endswith(".pdf"):
                continue

            if word.isupper():
                continue

            candidates.append(m)

        num_to_remove = int(len(candidates) * rate)

        if candidates and num_to_remove == 0:
            num_to_remove = 1

        selected = set(
            random.sample(
                range(len(candidates)),
                min(num_to_remove, len(candidates))
            )
        )

        new_line = line

        for i, match in enumerate(candidates):

            if i not in selected:
                continue

            start, end = match.start(), match.end()

            # удаляем слово + возможный лишний пробел
            new_line = new_line[:start] + "" + new_line[end:]

        result_lines.append(new_line)

    return "\n".join(result_lines)

In [59]:
df["word_deletion_text"] = [
    random_word_deletion(text)
    for text in df["Question"]]
df

,Question,Source Docs,Question Type,Source Chunk Type,Answer,spelling_errors_text,keyboard_typos_text,word_deletion_text
0,How has Apple's total net sales changed over t...,*AAPL*,Multi-Doc RAG,Table,"Based on the provided documents, Apple's total...",How has Apple's totul net sales changid over t...,How has Apple's total net sales changed over r...,How has Apple's net sales changed over time?
1,What are the major factors contributing to the...,*AAPL*,Multi-Doc RAG,Text,In the most recent 10-Q for the quarter ended ...,Whit are the major factors contributin to the ...,What are the kajor factors contributing to the...,What are the major factors contributing to the...
2,Has there been any significant change in Apple...,*AAPL*,Multi-Doc RAG,Table,"Yes, there has been a change in Apple's operat...",Has thure been any significant chinge in Apple...,Has there heen any significant change in Apple...,Has there been any significant change in 's op...
3,How has Apple's revenue from iPhone sales fluc...,*AAPL*,Multi-Doc RAG,Table,The revenue from iPhone sales for Apple has fl...,How has Apple's revenue frum ifone sales fluct...,How has Apple's fevenue from iPhone sales fluc...,How has Apple's revenue from sales fluctuated...
4,Can any trends be identified in Apple's Servic...,*AAPL*,Multi-Doc RAG,Table,"Based on the provided documents, there is a tr...",Can any trends be identified in Apple's Servac...,Can any trends be kdentified in Apple's Servic...,Can any trends be identified in Apple's Servic...
...,...,...,...,...,...,...,...,...
190,"For Amazon's Q1 2023 10-Q, align the details o...",*2023 Q1 AMZN*,Single-Doc Multi-Chunk RAG,Table,"In Amazon's Q1 2023 10-Q, the details of debt ...","For Amazan's Q1 2023 10-Q, align the details o...","For Amazon's Q1 2023 10-Q, slign the details o...","For Amazon's Q1 2023 10-Q, align the of debt ..."
191,Analyze how Amazon's effective tax rate report...,*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The effective tax rate for Amazon as reported ...,Anolyze how Amazon's effective tax rite report...,Analyze how Smazon's effective tax rate report...,how Amazon's effective tax rate reported in t...
192,"From Amazon's Q3 2023 10-Q, how does the opera...",*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The operational expenses section in Amazon's Q...,"Frum Amazon's Q3 2023 10-Q, how does the ipera...","From Smazon's Q3 2023 10-Q, how does the opera...","From Amazon's Q3 2023 10-Q, how does the opera..."
193,"In the latest 10-Q, how does the revenue from ...",*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The latest 10-Q does not provide specific info...,"In the litest 10-Q, how does the revenue fram ...","In the latest 10-Q, how xoes the revenue from ...","In the latest 10-Q, how does the from Amazon'..."


### Перестановка токенов  (stage 4)

In [12]:
import random
import re
from typing import List, Optional


def simple_tokenize(text: str) -> List[str]:
    """
    Простая токенизация:
    - слова
    - знаки пунктуации отдельными токенами
    """
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def det_shuffle(tokens: List[str], seed: int) -> List[str]:
    """
    Полная перестановка токенов (детерминированная).
    """
    rng = random.Random(seed)
    new_tokens = tokens[:]
    rng.shuffle(new_tokens)
    return new_tokens


def partial_shuffle(tokens: List[str], noise_level: float, seed: int) -> List[str]:
    """
    Частичная перестановка:
    - выбирается subset токенов и они перемешиваются между собой.
    
    noise_level = 0.0..1.0
    """
    if not (0.0 <= noise_level <= 1.0):
        raise ValueError("noise_level должен быть в диапазоне [0, 1]")

    rng = random.Random(seed)
    n = len(tokens)

    k = max(2, int(n * noise_level)) if n >= 2 else n
    indices = rng.sample(range(n), k)

    shuffled = indices[:]
    rng.shuffle(shuffled)

    new_tokens = tokens[:]
    for old_i, new_i in zip(indices, shuffled):
        new_tokens[old_i] = tokens[new_i]

    return new_tokens


def block_shuffle(tokens: List[str], block_size: int, seed: int) -> List[str]:
    """
    Разбивает токены на блоки block_size и перемешивает блоки местами.
    Это часто выглядит как человеческий "телеграфный стиль".
    """
    if block_size < 1:
        raise ValueError("block_size должен быть >= 1")

    rng = random.Random(seed)

    blocks = [tokens[i:i + block_size] for i in range(0, len(tokens), block_size)]
    rng.shuffle(blocks)

    result = []
    for b in blocks:
        result.extend(b)

    return result


def local_swap(tokens: List[str], swap_prob: float, seed: int) -> List[str]:
    """
    Локальные перестановки (похоже на быструю печать):
    с вероятностью swap_prob меняет местами соседние токены.
    """
    if not (0.0 <= swap_prob <= 1.0):
        raise ValueError("swap_prob должен быть в диапазоне [0, 1]")

    rng = random.Random(seed)
    new_tokens = tokens[:]

    i = 0
    while i < len(new_tokens) - 1:
        if rng.random() < swap_prob:
            new_tokens[i], new_tokens[i + 1] = new_tokens[i + 1], new_tokens[i]
            i += 2
        else:
            i += 1

    return new_tokens


def detokenize(tokens: List[str]) -> str:
    """
    Обратная сборка строки:
    - пробелы ставятся между словами
    - пунктуация приклеивается к слову слева
    """
    text = ""
    for t in tokens:
        if re.match(r"[^\w\s]", t):  # punctuation
            text += t
        else:
            if text and not text.endswith(" "):
                text += " "
            text += t
    return text.strip()


def add_token_permutation_noise(
    query: str,
    mode: str = "partial",
    seed: int = 42,
    noise_level: float = 0.4,
    block_size: int = 2,
    swap_prob: float = 0.3,
) -> str:
    """
    Генератор шума перестановкой токенов.
    
    mode:
    - "full"     : полная shuffle
    - "partial"  : shuffle части токенов
    - "block"    : shuffle блоков
    - "swap"     : локальные swap соседей
    """
    tokens = simple_tokenize(query)

    if len(tokens) < 2:
        return query

    if mode == "full":
        noisy = det_shuffle(tokens, seed)

    elif mode == "partial":
        noisy = partial_shuffle(tokens, noise_level=noise_level, seed=seed)

    elif mode == "block":
        noisy = block_shuffle(tokens, block_size=block_size, seed=seed)

    elif mode == "swap":
        noisy = local_swap(tokens, swap_prob=swap_prob, seed=seed)

    else:
        raise ValueError(f"Unknown mode: {mode}")

    return detokenize(noisy)

In [13]:
q = "Who wrote the book War and Peace?"
print(add_token_permutation_noise(q, mode="swap", swap_prob=0.6, seed=1))

wrote Who the book and War? Peace


In [14]:
df["token_permutation_text"] = [
    add_token_permutation_noise(text, mode="swap", swap_prob=0.6, seed=1)
    for text in df["Question"]]

In [19]:
df.to_csv('./df_stage_4.csv', index=False)
df

,Question,Source Docs,Question Type,Source Chunk Type,Answer,spelling_errors_text,keyboard_typos_text,word_deletion_text,token_permutation_text
0,How has Apple's total net sales changed over t...,*AAPL*,Multi-Doc RAG,Table,"Based on the provided documents, Apple's total...",How has Apple's totul net sales changid over t...,How has Apple's total net sales changed over r...,How has Apple's net sales changed over time?,has How Apple' total s sales net over changed ...
1,What are the major factors contributing to the...,*AAPL*,Multi-Doc RAG,Text,In the most recent 10-Q for the quarter ended ...,Whit are the major factors contributin to the ...,What are the kajor factors contributing to the...,What are the major factors contributing to the...,are What the major contributing factors the to...
2,Has there been any significant change in Apple...,*AAPL*,Multi-Doc RAG,Table,"Yes, there has been a change in Apple's operat...",Has thure been any significant chinge in Apple...,Has there heen any significant change in Apple...,Has there been any significant change in 's op...,there Has been any change significant Apple in...
3,How has Apple's revenue from iPhone sales fluc...,*AAPL*,Multi-Doc RAG,Table,The revenue from iPhone sales for Apple has fl...,How has Apple's revenue frum ifone sales fluct...,How has Apple's fevenue from iPhone sales fluc...,How has Apple's revenue from sales fluctuated...,has How Apple' revenue s iPhone from fluctuate...
4,Can any trends be identified in Apple's Servic...,*AAPL*,Multi-Doc RAG,Table,"Based on the provided documents, there is a tr...",Can any trends be identified in Apple's Servac...,Can any trends be kdentified in Apple's Servic...,Can any trends be identified in Apple's Servic...,any Can trends be in identified' Apple Service...
...,...,...,...,...,...,...,...,...,...
190,"For Amazon's Q1 2023 10-Q, align the details o...",*2023 Q1 AMZN*,Single-Doc Multi-Chunk RAG,Table,"In Amazon's Q1 2023 10-Q, the details of debt ...","For Amazan's Q1 2023 10-Q, align the details o...","For Amazon's Q1 2023 10-Q, slign the details o...","For Amazon's Q1 2023 10-Q, align the of debt ...","Amazon For' s 2023 Q1- 10, Q align the of deta..."
191,Analyze how Amazon's effective tax rate report...,*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The effective tax rate for Amazon as reported ...,Anolyze how Amazon's effective tax rite report...,Analyze how Smazon's effective tax rate report...,how Amazon's effective tax rate reported in t...,how Analyze Amazon' effective s rate tax in re...
192,"From Amazon's Q3 2023 10-Q, how does the opera...",*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The operational expenses section in Amazon's Q...,"Frum Amazon's Q3 2023 10-Q, how does the ipera...","From Smazon's Q3 2023 10-Q, how does the opera...","From Amazon's Q3 2023 10-Q, how does the opera...","Amazon From' s 2023 Q3- 10, Q how does operati..."
193,"In the latest 10-Q, how does the revenue from ...",*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The latest 10-Q does not provide specific info...,"In the litest 10-Q, how does the revenue fram ...","In the latest 10-Q, how xoes the revenue from ...","In the latest 10-Q, how does the from Amazon'...","the In latest 10 Q- how, the does revenue from..."


## Выделение списка литературы

In [22]:
import re

def extract_sources(text: str):
    # Берём всё после SOURCE(S):
    match = re.search(r"SOURCE\(S\):\s*(.+)$", text, flags=re.MULTILINE | re.DOTALL)
    if not match:
        return []

    sources_raw = match.group(1).strip()

    # Разбиваем по запятым
    parts = [p.strip() for p in sources_raw.split(",") if p.strip()]

    cleaned_sources = []
    for p in parts:
        # убираем кавычки, скобки, точки в конце
        p = re.sub(r'^[\s"\(\[\{]+', "", p)   # слева
        p = re.sub(r'[\s"\)\]\}\.]+$', "", p) # справа

        # убираем лишние пробелы внутри
        p = re.sub(r"\s+", " ", p).strip()

        if p:
            cleaned_sources.append(p)

    return cleaned_sources

In [30]:
df['listed_sources'] = [
    extract_sources(i)
    for i in df['Answer']]

## Реализация нормализации

### Seq2Seq

### Prompt-based

In [1]:
from ollama import chat
from ollama import Client
from ollama import ChatResponse

In [2]:
def ask_llm(system_prompt: str,
            text: str,
            host='http://10.0.0.54:11434',
            model='qwen3:30b',
            temperature = 0.0):
        
        client=Client(host=host)
        response: ChatResponse = client.chat(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": f"""
                        CONTEXT:
                        {text}
                    """
                }
            ],
            options={
                "temperature": temperature,   # critical
                "top_p": 0.8,
                "repeat_penalty": 1.1,
            },
            # Время жизни модели в памяти, если -1 то должна жить всегда
            keep_alive=-1
        )
        return response.message.content

In [3]:
import requests
from tqdm import tqdm

def ask_lightrag(query: str,
                 endpoint: str = 'http://localhost:9621/query',
                 mode = 'mix',
                 top_k: int = 10,
                 include_references = True):
    json_ = {
            "query": query,
            "mode": mode,
            "top_k": top_k,
            "include_references": include_references
            }
    
    req = requests.post(endpoint, json=json_)
    if req.status_code == 200:
        return req
    return -1

In [4]:
def parse_response(data: dict):
    """
    Извлекает response и уникальные references из json/dict.

    Returns:
        response_text (str)
        references_list (list)
    """

    response_text = data.get("response", "")

    # Удаление дублей с сохранением порядка
    references_list = list(dict.fromkeys(
        ref.get("file_path")
        for ref in data.get("references", [])
        if ref.get("file_path")
    ))

    return response_text, references_list

### Нормальный запрос без нормолизации

In [22]:
# responses = []
# references_all = []
# for idx, row  in tqdm(df.iterrows(), total=len(df['Question'])):
#     ans = ask_lightrag(row['Question'])
#     response_text, references = parse_response(ans.json())
#     responses.append(response_text)
#     references_all.append(references)

# df['response_real_question'] = responses
# df['references_real_question'] = references_all  

    

100%|██████████| 195/195 [4:21:01<00:00, 80.32s/it]   


In [ ]:
# # Первый проход без изменений и нормолизации
# df.to_csv('./df_etap_bez_norm_i_errors.csv', index=False)

### Запрос с первой ошибкой

In [27]:
# responses = []
# references_all = []
# for idx, row  in tqdm(df.iterrows(), total=len(df['spelling_errors_text'])):
#     ans = ask_lightrag(row['spelling_errors_text'])
#     response_text, references = parse_response(ans.json())
#     responses.append(response_text)
#     references_all.append(references)

# df['response_spelling_errors_text'] = responses
# df['references_spelling_errors_text'] = references_all  

100%|██████████| 195/195 [4:28:59<00:00, 82.77s/it]  


In [28]:
# responses = []
# references_all = []
# for idx, row  in tqdm(df.iterrows(), total=len(df['keyboard_typos_text'])):
#     ans = ask_lightrag(row['keyboard_typos_text'])
#     response_text, references = parse_response(ans.json())
#     responses.append(response_text)
#     references_all.append(references)

# df['response_keyboard_typos_text'] = responses
# df['references_keyboard_typos_text'] = references_all  

100%|██████████| 195/195 [4:27:45<00:00, 82.39s/it]   


In [29]:
# responses = []
# references_all = []
# for idx, row  in tqdm(df.iterrows(), total=len(df['word_deletion_text'])):
#     ans = ask_lightrag(row['word_deletion_text'])
#     response_text, references = parse_response(ans.json())
#     responses.append(response_text)
#     references_all.append(references)

# df['response_word_deletion_text'] = responses
# df['references_word_deletion_text'] = references_all  

100%|██████████| 195/195 [4:25:29<00:00, 81.69s/it]  


In [30]:
# responses = []
# references_all = []
# for idx, row  in tqdm(df.iterrows(), total=len(df['token_permutation_text'])):
#     ans = ask_lightrag(row['token_permutation_text'])
#     response_text, references = parse_response(ans.json())
#     responses.append(response_text)
#     references_all.append(references)

# df['response_token_permutation_text'] = responses
# df['references_token_permutation_text'] = references_all 

100%|██████████| 195/195 [4:35:50<00:00, 84.87s/it]  


In [ ]:
# df.to_csv('./df_all_answers.csv', index=False)

## Метрики

In [ ]:
import re
import math
from typing import List
from collections import Counter
import math
from typing import List

# =========================
# Normalization
# =========================

def normalize_name(name: str) -> str:
    if isinstance(name, str):
        n = name.lower()
        n = re.sub(r'\.pdf$', '', n)
        n = re.sub(r'[^0-9a-z\s\-\_]', '', n)   # убрать странные символы
        n = re.sub(r'\s+', ' ', n).strip()
        return n
    return ''

def normalize_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'[^0-9a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_all_text(text: List[List[str]]) -> List[List[str]]:
    return [
        normalize_name(inner_list) for inner_list in text
    ]    

# =========================
# Precision@K
# =========================

def precision_at_k_single(retrieved: List[str], gold: List[str], k: int = 5) -> float:
    if not retrieved:
        return 0.0

    retrieved_k = set(retrieved[:k])
    gold_set = set(gold)

    hits = len(retrieved_k & gold_set)

    return hits / k


# =========================
# Recall@K
# =========================

def recall_at_k_single(retrieved: List[str], gold: List[str], k: int = 5) -> float:
    if not gold:
        return 0.0

    retrieved_k = set(retrieved[:k])
    gold_set = set(gold)

    hits = len(retrieved_k & gold_set)

    return hits / len(gold_set)


# =========================
# Hit@K
# =========================

def hit_at_k_single(retrieved: List[str], gold: List[str], k: int = 5) -> int:
    return int(any(doc in gold for doc in retrieved[:k]))


# =========================
# MRR
# =========================

def mrr_single(retrieved: List[str], gold: List[str]) -> float:
    for idx, doc in enumerate(retrieved):
        if doc in gold:
            return 1.0 / (idx + 1)
    return 0.0


# =========================
# nDCG@K
# =========================

def dcg_at_k(retrieved: List[str], gold: List[str], k: int = 5) -> float:
    dcg = 0.0
    for i, doc in enumerate(retrieved[:k]):
        rel = 1 if doc in gold else 0
        dcg += rel / math.log2(i + 2)
    return dcg


def ndcg_at_k_single(retrieved: List[str], gold: List[str], k: int = 5) -> float:
    ideal_rels = [1] * min(len(set(gold)), k)
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal_rels))

    if idcg == 0:
        return 0.0

    return dcg_at_k(retrieved, gold, k) / idcg


# =========================
# Batch версия
# =========================

def retrieval_batch(retrieved_batch: List[List[str]],
                    gold_batch: List[List[str]],
                    k: int = 5):

    precision_scores = []
    recall_scores = []
    hit_scores = []
    mrr_scores = []
    ndcg_scores = []

    for r, g in zip(retrieved_batch, gold_batch):
        precision_scores.append(precision_at_k_single(r, g, k))
        recall_scores.append(recall_at_k_single(r, g, k))
        hit_scores.append(hit_at_k_single(r, g, k))
        mrr_scores.append(mrr_single(r, g))
        ndcg_scores.append(ndcg_at_k_single(r, g, k))

    return {
        "Precision@K": sum(precision_scores) / len(precision_scores),
        "Recall@K": sum(recall_scores) / len(recall_scores),
        "Hit@K": sum(hit_scores) / len(hit_scores),
        "MRR": sum(mrr_scores) / len(mrr_scores),
        "nDCG@K": sum(ndcg_scores) / len(ndcg_scores)
    }

In [35]:
from collections import Counter


def exact_match_single(pred: str, gold: str) -> int:
    return int(pred == gold)

def f1_single(pred: str, gold: str) -> float:
    pred_tokens = pred.split()
    gold_tokens = gold.split()

    if not pred_tokens or not gold_tokens:
        return 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

def qa_batch(pred_batch: List[str], gold_batch: List[str]):
    em_scores = []
    f1_scores = []

    for p, g in zip(pred_batch, gold_batch):
        em_scores.append(exact_match_single(p, g))
        f1_scores.append(f1_single(p, g))

    return {
        "EM": sum(em_scores) / len(em_scores),
        "F1": sum(f1_scores) / len(f1_scores)
    }

In [36]:
# Подготавливаем данныые для расчета показателей
def calc_scores(df, gold_list: str, rag_list: str, rag_answer: str):
    gold_ans_ = [[normalize_name(x) for x in inner_list]
                    for inner_list in 
                    [i.split(',') for i in df[gold_list].fillna('').to_list()]]
    rag_ans_ = [[normalize_name(x) for x in inner_list]
                    for inner_list in 
                    [i.split(',') for i in df[rag_list].fillna('').to_list()]]
    norm_real_ans_ = normalize_all_text(df['Answer'].to_list())
    norm_rag_ans_ = normalize_all_text(df[rag_answer].to_list())
    print(retrieval_batch(rag_ans_, gold_ans_))
    print(qa_batch(norm_rag_ans_, norm_real_ans_))

In [51]:
import numpy as np

K = 10


def precision_at_k(real, pred, k=5):
    pred_k = pred[:k]

    if len(pred_k) == 0:
        return 0.0

    hits = len(set(pred_k) & set(real))
    return hits / k


def recall_at_k(real, pred, k=5):
    pred_k = pred[:k]

    if len(real) == 0:
        return 0.0

    hits = len(set(pred_k) & set(real))
    return hits / len(real)


def hit_at_k(real, pred, k=5):
    pred_k = pred[:k]
    return int(len(set(pred_k) & set(real)) > 0)


def mrr(real, pred):
    for idx, item in enumerate(pred, start=1):
        if item in real:
            return 1 / idx
    return 0.0


def dcg_at_k(real, pred, k=5):
    pred_k = pred[:k]

    dcg = 0.0
    for i, item in enumerate(pred_k, start=1):
        if item in real:
            dcg += 1 / np.log2(i + 1)

    return dcg


def ndcg_at_k(real, pred, k=5):
    dcg = dcg_at_k(real, pred, k)

    ideal_hits = min(len(real), k)

    idcg = sum(
        1 / np.log2(i + 1)
        for i in range(1, ideal_hits + 1)
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg

In [64]:
import ast

In [73]:
df["listed_sources"] = df["listed_sources"].apply(ast.literal_eval)

references_real_question            | +   
references_spelling_errors_text     | +   
references_keyboard_typos_text      | +    
references_word_deletion_text       | +   
references_token_permutation_text   | +   

In [84]:
where_ = 'references_token_permutation_text'


df[f"precision@10_{where_}"] = df.apply(
    lambda x: precision_at_k(
        x["listed_sources"],
        x[where_],
        k=10
    ),
    axis=1
)

df[f"recall@10_{where_}"] = df.apply(
    lambda x: recall_at_k(
        x["listed_sources"],
        x[where_],
        k=10
    ),
    axis=1
)

df[f"hit@10_real_{where_}"] = df.apply(
    lambda x: hit_at_k(
        x["listed_sources"],
        x[where_],
        k=10
    ),
    axis=1
)

df[f"mrr_{where_}"] = df.apply(
    lambda x: mrr(
        x["listed_sources"],
        x[where_]
    ),
    axis=1
)

df[f"ndcg@10_{where_}"] = df.apply(
    lambda x: ndcg_at_k(
        x["listed_sources"],
        x[where_],
        k=10
    ),
    axis=1
)

metrics = {
    "Precision@5": df[f"precision@10_{where_}"].mean(),
    "Recall@5": df[f"recall@10_{where_}"].mean(),
    "Hit@5": df[f"hit@10_real_{where_}"].mean(),
    "MRR": df[f"mrr_{where_}"].mean(),
    "nDCG@5": df[f"ndcg@10_{where_}"].mean(),
}

metrics

{'Precision@5': np.float64(0.1794871794871795),
 'Recall@5': np.float64(0.9525885225885226),
 'Hit@5': np.float64(0.9794871794871794),
 'MRR': np.float64(0.7012942612942614),
 'nDCG@5': np.float64(0.7480380647199599)}

In [86]:
from collections import Counter

def normalize_text(s):
    return s.lower().strip()


def token_f1(y_true, y_pred):
    true_tokens = normalize_text(y_true).split()
    pred_tokens = normalize_text(y_pred).split()

    common = Counter(true_tokens) & Counter(pred_tokens)

    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(true_tokens)

    return 2 * precision * recall / (precision + recall)

In [ ]:
response_real_question            | +   
response_spelling_errors_text     | +   
response_keyboard_typos_text      | +    
response_word_deletion_text       | +  
response_token_permutation_text   | -   

In [95]:
response_ = 'response_token_permutation_text'

df[f"f1_generation_{response_}"] = df.apply(
    lambda x: token_f1(
        x["Answer"],
        x[response_]
    ),
    axis=1
)
metrics = {
    "F1-score": df[f"f1_generation_{response_}"].mean(),
}
metrics

{'F1-score': np.float64(0.28074941421535227)}

In [103]:
import json
import requests


OLLAMA_URL = "http://10.0.0.54:11434/api/chat"


def llm_judge(
    real_answer: str,
    rag_answer: str,
    model: str = "qwen3:30b",
):

    system_prompt = """
You are an evaluator for RAG systems.

You must return ONLY valid JSON.

Evaluation criteria:
- faithfulness: factual correctness
- relevance: does the answer address the reference answer
- completeness: does it contain all important information

Scores must be integers from 1 to 10.

Return JSON only.
"""

    user_prompt = f"""
REFERENCE ANSWER:
{real_answer}

GENERATED ANSWER:
{rag_answer}

Return JSON in this exact format:

{{
  "score": 0,
  "faithfulness": 0,
  "relevance": 0,
  "completeness": 0,
  "explanation": ""
}}
"""

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "messages": [
                {
                    "role": "system",
                    "content": system_prompt,
                },
                {
                    "role": "user",
                    "content": user_prompt,
                },
            ],
            "stream": False,
        },
        timeout=300,
    )

    data = response.json()

    content = data["message"]["content"]

    try:
        return json.loads(content)

    except Exception:
        print("RAW RESPONSE:")
        print(content)

        return {
            "score": None,
            "faithfulness": None,
            "relevance": None,
            "completeness": None,
            "explanation": content,
        }

In [105]:
llm_judge(df['Answer'][0], df['response_real_question'][0])

{'score': 5,
 'faithfulness': 3,
 'relevance': 3,
 'completeness': 10,
 'explanation': 'Generated answer incorrectly states Q1 2023 sales as $94.84B (should be $117.15B), leading to inaccurate trend description. Q2 2023 number is correct but mislabeled as same as Q1.'}